In [1]:
import os
os.environ['http_proxy'] = "http://localhost:7890"
os.environ['https_proxy'] = "http://localhost:7890"
os.environ['all_proxy'] = "socks5://localhost:7890"

## (1) Load model

In [2]:
from model import Mamba, ModelArgs
from transformers import AutoTokenizer

# One of:
#     'state-spaces/mamba-2.8b-slimpj'
#     'state-spaces/mamba-2.8b'
#     'state-spaces/mamba-1.4b'
#     'state-spaces/mamba-790m'
#     'state-spaces/mamba-370m'
#     'state-spaces/mamba-130m'
pretrained_model_name = 'state-spaces/mamba-370m'

model = Mamba.from_pretrained(pretrained_model_name)
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')

/home/tianq/venv/lib/python3.12/site-packages/torch/xpu/__init__.py:61: UserWarning: XPU device count is zero! (Triggered internally at /pytorch/c10/xpu/XPUFunctions.cpp:115.)
  return torch._C._xpu_getDeviceCount()


config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.49G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

## (2) Generate Text

In [3]:
import torch
import torch.nn.functional as F


def generate(model,
             tokenizer,
             prompt: str,
             n_tokens_to_gen: int = 50,
             sample: bool = True,
             top_k: int = 40):
    model.eval()
    
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids
    
    for token_n in range(n_tokens_to_gen):
        with torch.no_grad():
            indices_to_input = input_ids
            next_token_logits = model(indices_to_input)[:, -1]
        
        probs = F.softmax(next_token_logits, dim=-1)
        (batch, vocab_size) = probs.shape
        
        if top_k is not None:
            (values, indices) = torch.topk(probs, k=top_k)
            probs[probs < values[:, -1, None]] = 0
            probs = probs / probs.sum(axis=1, keepdims=True)
        
        if sample:
            next_indices = torch.multinomial(probs, num_samples=1)
        else:
            next_indices = torch.argmax(probs, dim=-1)[:, None]
        
        input_ids = torch.cat([input_ids, next_indices], dim=1)

    output_completions = [tokenizer.decode(output.tolist()) for output in input_ids][0]
    
    return output_completions

In [4]:
print(generate(model, tokenizer, 'Mamba is the'))

Mamba is the first project we're making without any ads, but with the support of people who support us."

In response to the negative reaction of his fans, the artist took to social media to defend himself: "It's about me, my beliefs,


In [5]:
print(generate(model, tokenizer, 'John: Hi!\nSally:'))

John: Hi!
Sally: Hello!
John: Hi.
Sally: Hi.
John: So what can you say about your first day on the job?
Sally: I really got into it!
John: Yeah.
Sally: I got


In [6]:
print(generate(model, tokenizer, 'The meaning of life is '))

The meaning of life is 「結師」-eraki- in Chinese. However, the Chinese people use 「結師」-eraki- to mean “success” in order to avoid confusion.”(2) This may be a misunderstanding.


In [7]:
print(generate(model, tokenizer, 'def reverse_string('))

def reverse_string(self, n):
        # reverse one character, like / is 3 -> /1
        return '%' + int(n, 10)
<|endoftext|>Q:

How to read a string into a list of integers?

I
